# MTBS + gridMET: Bootstrapped Multiple Linear Regression

**Target:** `log1p(burned acres)` for MTBS fire events. **Predictors:** pre-ignition weather from gridMET plus location and season. **No LANDFIRE data.**

Run from the project root in VS Code. Uses a chronological holdout and 300 bootstrap fits. This predicts the final area of **recorded MTBS fires**, not whether a fire will start. The MTBS inclusion thresholds mean smaller fires are underrepresented.

**Weather coverage matters:** Only fires with all required pre-ignition weather values are retained. Run the coverage cell before interpreting results. GridMET files may be stored as `data/raw/gridmet_yearly/VAR_YEAR.nc` or merged `data/raw/gridmet_START_END.nc`.

In [ ]:
# Run once if necessary in a notebook cell:
# %pip install geopandas pandas numpy xarray netCDF4 scipy scikit-learn matplotlib rasterio ipykernel

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RAW = Path('data/raw')
PROCESSED = Path('data/processed'); PROCESSED.mkdir(parents=True, exist_ok=True)
GRIDMET_VARS = ['pr', 'tmmx', 'rmin', 'vs', 'erc', 'fm100']
# Limit while testing; set to None to use all fire events with valid dates/geometries
MAX_EVENTS = 300
BOOTSTRAPS = 300
SEED = 42
rng = np.random.default_rng(SEED)
print('Working directory:', Path.cwd())

## 1. Load MTBS and create event coordinates
Uses burned-area attribute `burnbndac` (acres) and `ig_date`. One record may not correspond to every ignition; we use MTBS event records as published. For polygons, use representative points to ensure coordinates lie inside the perimeter.

In [ ]:
gpkg_files = sorted(RAW.glob('mtbs_*.gpkg'))
shp_file = RAW / 'mtbs' / 'mtbs_perims_DD.shp'
if gpkg_files:
    # Prefer latest modified processed regional extraction if several exist
    source = max(gpkg_files, key=lambda p: p.stat().st_mtime)
elif shp_file.exists():
    source = shp_file
else:
    raise FileNotFoundError('No MTBS .gpkg or mtbs_perims_DD.shp found under data/raw')
print('MTBS source:', source)
fires = gpd.read_file(source)
need = ['ig_date', 'burnbndac', 'geometry']
missing = [c for c in need if c not in fires]
if missing: raise ValueError(f'Missing required MTBS columns: {missing}')
fires['ig_date'] = pd.to_datetime(fires['ig_date'], errors='coerce')
fires['burnbndac'] = pd.to_numeric(fires['burnbndac'], errors='coerce')
fires = fires.dropna(subset=['ig_date','burnbndac','geometry']).copy()
fires = fires[(fires['burnbndac'] > 0) & (~fires.geometry.is_empty)].copy()
if fires.crs is None: raise ValueError('MTBS geometry CRS is missing; cannot safely assign weather coordinates.')
# Calculate representative points in a projected CRS before conversion to lon/lat
pts = fires.to_crs('EPSG:5070').representative_point().to_crs('EPSG:4326')
fires['longitude'] = pts.x
fires['latitude'] = pts.y
fires['year'] = fires['ig_date'].dt.year
fires['month'] = fires['ig_date'].dt.month
if MAX_EVENTS is not None and len(fires)>MAX_EVENTS:
    # Sample across years, with a fixed seed for repeatability
    fires = fires.sample(n=MAX_EVENTS, random_state=SEED).sort_values('ig_date')
print('Fire records:', len(fires), 'year range:', (fires.year.min(), fires.year.max()))
display(fires[['ig_date','burnbndac','latitude','longitude']].head())

## 2. Inspect gridMET availability
We require up to 30 days **before** each ignition. To cover January fires, download the preceding year's weather as well. This notebook does not download multi-gigabyte files automatically.

In [ ]:
yearly_dir = RAW/'gridmet_yearly'
merged_paths = sorted(RAW.glob('gridmet_????-??-??_????-??-??.nc'))
yearly_paths = sorted(yearly_dir.glob('*.nc')) if yearly_dir.exists() else []
print('Yearly .nc files:', len(yearly_paths))
print('Merged .nc files:', [p.name for p in merged_paths])
print('Examples:', [p.name for p in yearly_paths[:8]])
if not yearly_paths and not merged_paths:
    raise FileNotFoundError('gridMET NetCDF missing. Uncomment download_gridmet(...) in scripts/download_raw_data.py and run that script.')

## 3. Extract pre-fire weather
- Temperature (gridMET `tmmx`) is in Kelvin → converted to Celsius.
- Precipitation (`pr`) summed over days −30 to −1.
- `rmin`, `vs`, `erc`, `fm100` averaged over days −7 to −1.

A weather summary is only calculated if its complete time window is present. Time coordinates may be called `day` or `time`, and longitude conventions are adjusted. Data are sampled at the closest grid cell. To keep memory low, files stay lazily open and only sampled values are read.

In [ ]:
# Load each variable's yearly file, or fall back to matching variables in merged NetCDFs.
# Keeping datasets open makes repeated point sampling practical; close them at the end.
opened = []
cache = {}

def open_weather(var, year):
    key=(var,year)
    if key in cache: return cache[key]
    path=yearly_dir/f'{var}_{year}.nc'
    if path.exists():
        ds=xr.open_dataset(path)
    else:
        ds=None
        for f in merged_paths:
            candidate=xr.open_dataset(f)
            names=[x for x in candidate.data_vars if x != 'crs']
            # Single-variable merged name or native gridMET variable abbreviations
            if var in candidate.data_vars:
                ds=candidate
                break
            candidate.close()
        if ds is None:
            cache[key]=None
            return None
    time_name=next((x for x in ('day','time') if x in ds.coords), None)
    if time_name is None:
        ds.close(); cache[key]=None; return None
    names=[x for x in ds.data_vars if x!='crs']
    actual=var if var in names else (names[0] if len(names)==1 else None)
    if actual is None:
        ds.close(); cache[key]=None; return None
    opened.append(ds)
    cache[key]=(ds, actual, time_name)
    return cache[key]

def window_avg(var, lon, lat, end_date, days, agg='mean'):
    # end_date is ignition day; exclude that day to prevent leakage
    stop=pd.Timestamp(end_date).normalize()-pd.Timedelta(days=1)
    start=stop-pd.Timedelta(days=days-1)
    blocks=[]
    for year in range(start.year, stop.year+1):
        info=open_weather(var,year)
        if info is None: return np.nan
        ds, name, dim=info
        try:
            xlon = lon % 360 if float(ds.lon.max()) > 180 else lon
            cell=ds[name].sel(lon=xlon,lat=lat,method='nearest')
            part=cell.sel({dim:slice(str(start.date()),str(stop.date()))})
            if part.sizes.get(dim,0):
                s=part.to_series()
                blocks.append(s)
        except (KeyError, ValueError, IndexError):
            return np.nan
    if not blocks: return np.nan
    series=pd.concat(blocks)
    series.index=pd.to_datetime(series.index).normalize()
    series=series[~series.index.duplicated(keep='first')]
    required=pd.date_range(start,stop,freq='D')
    series=series.reindex(required)
    if series.isna().any(): return np.nan
    value=float(series.sum() if agg=='sum' else series.mean())
    if var=='tmmx': value-=273.15 # gridMET tmmx Kelvin
    return value

FEATURE_WINDOWS = {
    'tmax_7d_c': ('tmmx',7,'mean'),
    'rmin_7d_pct': ('rmin',7,'mean'),
    'wind_7d_ms': ('vs',7,'mean'),
    'erc_7d': ('erc',7,'mean'),
    'fm100_7d': ('fm100',7,'mean'),
    'precip_30d_mm': ('pr',30,'sum')
}
records=[]
for i, (_, fire) in enumerate(fires.iterrows(),1):
    record={'latitude':fire.latitude,'longitude':fire.longitude,
            'ig_date':fire.ig_date,'year':fire.year,'month':fire.month,
            'burned_acres':float(fire.burnbndac)}
    for output,(var,days,agg) in FEATURE_WINDOWS.items():
        record[output]=window_avg(var,fire.longitude,fire.latitude,fire.ig_date,days,agg)
    records.append(record)
    if i%50==0: print(f'Weather matched for {i}/{len(fires)} fires')
weather_fires=pd.DataFrame.from_records(records)
for ds in opened: ds.close()
cache.clear()
print('Records with missing weather:',weather_fires[list(FEATURE_WINDOWS)].isna().any(axis=1).sum())
display(weather_fires.isna().sum().rename('missing values').to_frame())
weather_fires.to_csv(PROCESSED/'mtbs_gridmet_merged.csv',index=False)

## 4. Select complete training rows and chronological holdout
A small or unrepresentative matched sample is not evidence of predictive performance. To avoid accidentally fitting a weather-free model, the notebook requires **all six weather features**. It also requires at least two years of matching data and enough examples for a useful holdout. Do not replace missing weather with zero.

In [ ]:
WEATHER_COLS=list(FEATURE_WINDOWS)
model_df=weather_fires.dropna(subset=WEATHER_COLS).copy()
model_df['log_acres']=np.log1p(model_df['burned_acres'])
model_df['month_sin']=np.sin(2*np.pi*model_df['month']/12)
model_df['month_cos']=np.cos(2*np.pi*model_df['month']/12)
FEATURES=['latitude','longitude','year','month_sin','month_cos']+WEATHER_COLS
model_df=model_df.sort_values('ig_date').reset_index(drop=True)
print(f'Complete weather coverage: {len(model_df)}/{len(weather_fires)} fires')
print('By year:'); display(model_df.groupby('year').size().rename('events').to_frame())
if len(model_df)<40 or model_df['year'].nunique()<2:
    raise ValueError('Not enough complete weather-matched fires across >=2 years. Download gridMET for the MTBS fire years (and preceding years for January).')
# Hold out complete latest year(s), not random rows within the same year
unique_years=np.sort(model_df.year.unique())
cut_year=unique_years[max(1,int(np.floor(len(unique_years)*0.8)))]
train=model_df[model_df.year<cut_year].copy()
test=model_df[model_df.year>=cut_year].copy()
if len(train)<20 or len(test)<10:
    raise ValueError(f'Chronological split too small ({len(train)} train/{len(test)} test); collect more years.')
X_train=train[FEATURES]; X_test=test[FEATURES]
y_train=train.log_acres; y_test=test.log_acres
print('Train years:',train.year.min(),'-',train.year.max(), 'n=',len(train))
print('Test years:',test.year.min(),'-',test.year.max(), 'n=',len(test))

## 5. Fit and bootstrap linear regression
Resample **training fire records** with replacement. Model uncertainty intervals describe variability in fitted mean predictions, **not** prediction intervals for an individual future wildfire. For robust temporal uncertainty estimates, consider block bootstrapping by year or region rather than independent fires.

In [ ]:
model=LinearRegression().fit(X_train,y_train)
base_pred=model.predict(X_test)
boot_predictions=np.empty((BOOTSTRAPS,len(X_test)))
boot_coefs=np.empty((BOOTSTRAPS,len(FEATURES)))
for b in range(BOOTSTRAPS):
    indices=rng.integers(0,len(X_train),len(X_train))
    sampled=LinearRegression().fit(X_train.iloc[indices],y_train.iloc[indices])
    boot_predictions[b]=sampled.predict(X_test)
    boot_coefs[b]=sampled.coef_
low,high=np.quantile(boot_predictions,[0.025,0.975],axis=0)
print('MAE (log acres):',round(mean_absolute_error(y_test,base_pred),3))
print('RMSE (log acres):',round(np.sqrt(mean_squared_error(y_test,base_pred)),3))
print('R² (log acres):',round(r2_score(y_test,base_pred),3))
print('MAE (acres):',round(mean_absolute_error(np.expm1(y_test),np.expm1(base_pred)),1))
coef_summary=pd.DataFrame({'feature':FEATURES,'coefficient':model.coef_,
                           'CI_lower':np.quantile(boot_coefs,.025,axis=0),
                           'CI_upper':np.quantile(boot_coefs,.975,axis=0)})
display(coef_summary)
results=test[['ig_date','year','latitude','longitude','burned_acres']].copy()
results['predicted_log_acres']=base_pred
results['predicted_acres']=np.expm1(base_pred)
results['bootstrap_mean_CI_low_log']=low
results['bootstrap_mean_CI_high_log']=high
results.to_csv(PROCESSED/'mtbs_gridmet_bootstrap_predictions.csv',index=False)
print('Saved:',PROCESSED/'mtbs_gridmet_bootstrap_predictions.csv')

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].scatter(y_test,base_pred,alpha=.35,s=15)
lims=[min(y_test.min(),base_pred.min()),max(y_test.max(),base_pred.max())]
axes[0].plot(lims,lims,'k--',label='Perfect prediction')
axes[0].set(xlabel='Observed log(1 + acres)',ylabel='Predicted log(1 + acres)',title='Held-out fires: MTBS + gridMET')
axes[0].legend()
axes[1].scatter(base_pred,y_test-base_pred,alpha=.35,s=15)
axes[1].axhline(0,ls='--',color='black')
axes[1].set(xlabel='Predicted log(1 + acres)',ylabel='Residual (actual - predicted)',title='Residuals')
plt.tight_layout();plt.show()

## Interpretation and caveats
- Compare these metrics against the **MTBS-only baseline on the exact same matched rows and same test years**. Otherwise, differences could come from changing the sample rather than adding gridMET.
- MTBS records generally represent fires above mapping thresholds; this is **conditional fire-size regression**, not ignition probability.
- GridMET is gridded weather, not exact weather at each fire. Fire perimeter representative points are an approximation of ignition location; use actual ignition points if available.
- A linear model may underpredict large fires. Weather alone omits suppression, fire duration, terrain, fuel loading and other drivers.
- `year` can encode secular trends, but also makes extrapolation hard. Compare fit with and without it.
- For all-US analysis, download weather covering **the relevant fire years and geography**. Your previous `download_gridmet(..., 'east')` subset won't cover western fires.
- Avoid using severity or other post-ignition-derived fields as predictors.
